# Run any PAI script on Kaggle (1x or 2x T4)

**Settings:** Accelerator = *GPU T4 x2*, Internet = *on*. Run with **Save Version -> Save & Run All**
so the job continues after you close the browser. Everything in `/kaggle/working` becomes this
version's output.

Set `SCRIPT` and `ARGS` below, e.g.
- `scripts/slice_collect.py` with `--config configs/tabletop.yaml --episodes 3000 --workers 4`
- `scripts/slice_train_wm.py` with `--config configs/tabletop.yaml world_model.steps=20000`
- `scripts/collect_frames.py`, then `scripts/train_slots.py --extract`, then `scripts/train_slots.py`

**Resuming:** training scripts resume from `ckpt_last.pt` in their output folder. To continue in a
new session, attach the previous version's output (*Add Input -> Your Work*) and set `RESTORE_FROM`
to its folder; it is copied back into place before the script starts.
Each account owner runs their own copy. The commit hash is logged in every checkpoint.

In [ ]:
import os
REPO_URL = "https://github.com/<you>/pixel-active-inference.git"   # <- set this
COMMIT = "main"                                                     # pin a commit for reported runs
SCRIPT = "scripts/slice_train_wm.py"
ARGS = "--config configs/tabletop.yaml"
RESTORE_FROM = ""   # e.g. /kaggle/input/<previous-version>/pai  (copies runs/ and data/ back)
os.environ["MUJOCO_GL"] = "egl"   # headless GPU rendering

In [ ]:
!git clone -q {REPO_URL} /kaggle/working/pai && git -C /kaggle/working/pai checkout -q {COMMIT}
%cd /kaggle/working/pai
!pip install -q -e .
!python scripts/fetch_assets.py
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
if RESTORE_FROM:
    for sub in ("runs", "data"):
        src = os.path.join(RESTORE_FROM, sub)
        if os.path.isdir(src):
            !cp -rn {src} .
!python {SCRIPT} {ARGS}

In [ ]:
!ls -R runs | head -50